# Deliverable 5 — Production physics sweep under local compute budget

Measures t=1, all requested Reynolds and Mach values, five timing repetitions at N=32, plus N-convergence points through N=64. N=128–2048 remain scale-up runs because the existing N=256 point costs about 510 seconds per repetition.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import json, platform, numpy as np, pandas as pd
from quantum_aero.classical import LBMConfig, run_lbm
from quantum_aero.advanced import extended_lbm_diagnostics

rows=[]
for re in (10,100,400,1000,2000,5000):
 for ma in (.1,.05,.025,.0125):
  cfg=LBMConfig(n=32,reynolds=re,t_end=1,mach=ma,snapshots=2)
  diag=extended_lbm_diagnostics(cfg)
  timings=[diag["runtime_seconds"]]+[run_lbm(cfg)["runtime_seconds"] for _ in range(4)]
  rows.append({"reynolds":re,"n":32,"mach":ma,**diag,
               "runtime_median_seconds":float(np.median(timings)),"runtime_min_seconds":min(timings),"runtime_max_seconds":max(timings)})
df=pd.DataFrame(rows); df.to_csv(output_dir/"09_production_physics_n32.csv",index=False)
df[["reynolds","mach","relative_l2","pressure_relative_l2","fourier_mode_relative_error","minimum_population","runtime_median_seconds"]]

,reynolds,mach,relative_l2,pressure_relative_l2,fourier_mode_relative_error,minimum_population,runtime_median_seconds
0,10,0.1000,0.005421,0.755107,0.007004,0.024071,0.671232
1,10,0.0500,0.004635,0.362961,0.007157,0.025875,0.857731
2,10,0.0250,0.003820,0.558862,0.007188,0.026814,1.578255
3,10,0.0125,0.003675,0.313708,0.007186,0.027293,3.062620
4,100,0.1000,0.006936,0.756790,0.007118,0.023761,0.385732
5,100,0.0500,0.005711,0.363923,0.007136,0.025710,0.770805
6,100,0.0250,0.004451,0.558542,0.007188,0.026729,1.535794
7,100,0.0125,0.004163,0.313435,0.007093,0.027250,3.138774
8,400,0.1000,0.006944,0.756792,0.006613,0.023735,0.397499
9,400,0.0500,0.005643,0.363780,0.006667,0.025693,0.780295


In [3]:
conv=[]
for re in (10,100,400,1000,2000,5000):
 for n in (16,32,64):
  d=extended_lbm_diagnostics(LBMConfig(n=n,reynolds=re,t_end=1,mach=.025,snapshots=2))
  conv.append({"reynolds":re,"n":n,**d})
conv_df=pd.DataFrame(conv)
orders=[]
for re,g in conv_df.groupby("reynolds"):
 g=g.sort_values("n"); e=g.relative_l2.to_numpy(); ns=g.n.to_numpy()
 orders.append({"reynolds":re,"order_16_32":np.log(e[0]/e[1])/np.log(2),"order_32_64":np.log(e[1]/e[2])/np.log(2)})
order_df=pd.DataFrame(orders)
conv_df.to_csv(output_dir/"09_production_physics_convergence.csv",index=False); order_df.to_csv(output_dir/"09_convergence_orders.csv",index=False)
order_df

,reynolds,order_16_32,order_32_64
0,10,1.911508,1.829410
1,100,1.811151,1.760780
2,400,1.805259,1.725509
3,1000,1.815286,1.700686
4,2000,1.820015,1.685459
5,5000,1.823171,1.672535


In [4]:
meta={"python":platform.python_version(),"platform":platform.platform(),"processor":platform.processor(),"measured_rows":len(df)+len(conv_df),
"unmeasured_grids":[128,256,512,1024,2048],"reason":"local time/memory budget; existing N=256 t=1 measurement is ~510 s for one repetition"}
(output_dir/"09_production_physics_metadata.json").write_text(json.dumps(meta,indent=2))
assert len(df)==24 and len(conv_df)==18
print("PASS: requested Re/Ma/t/repetition sweep completed at N=32; resolution extension measured through N=64.")

PASS: requested Re/Ma/t/repetition sweep completed at N=32; resolution extension measured through N=64.
